# CSIP Module 08: Deep Learning & Neural Network Benchmarks
**Computational Society Intelligence Platform (CSIP) — Behavioral Decision Intelligence (BDI)**

### SKKNI Data Science Competency Standards
- **[UK-09 Membangun Model]**: Penerapan arsitektur Deep Learning (Tabular MLP dengan Entity Embeddings dan NLP Text Bi-LSTM).
- **Modul 14 & 15**: Multi-Layer Perceptron (MLP), Batch Normalization, Dropout, Word Embeddings, Bidirectional LSTM.

---

In [ ]:
import sys
from pathlib import Path

# Set repo root
REPO_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from csip.models.deep_tabular import TabularMLPClassifier, TabularEmbeddingMLP
from csip.models.deep_seq import TextBiLSTMClassifier

print(f'PyTorch Version: {torch.__version__}')
print(f'Device available: cpu (Threads: {torch.get_num_threads()})')

## 1. Load ML Leaderboard & Deep Learning Metrics
We inspect the comparative performance of Deep Learning models (B4) against the Classical ML Baselines (B0–B3).

In [ ]:
leaderboard_path = REPO_ROOT / 'artifacts' / 'reports' / 'ml_leaderboard.json'
with open(leaderboard_path, 'r') as f:
    leaderboard = json.load(f)

ldf = pd.DataFrame(leaderboard)
review_models = ldf[ldf['target'] == 'negative_review_flag'].drop_duplicates(subset=['model_name'])
display(review_models[['model_name', 'baseline_tier', 'test_pr_auc', 'test_roc_auc', 'test_f1', 'test_brier']])

## 2. Tabular MLP vs Classical Gradient Boosting
- **Finding**: LightGBM achieves PR-AUC = 0.3004, while Tabular MLP with Entity Embeddings achieves PR-AUC = 0.2875.
- **Empirical Insight**: Gradient boosted trees continue to dominate tabular structured data with heterogeneous numerical and categorical distributions on moderate sample sizes (~70k), while Deep Learning demonstrates solid competitive performance with continuous embeddings.

In [ ]:
# Inspect Tabular MLP Saved Checkpoint
mlp_ckpt = torch.load(REPO_ROOT / 'artifacts' / 'models' / 'tabular_mlp_negative_review.pt', weights_only=False)
print('Tabular MLP Architecture Summary:')
print(f"- Continuous features: {mlp_ckpt['num_continuous']}")
print(f"- Embedding sizes: {mlp_ckpt['embedding_sizes']}")
print(f"- Hidden dimensions: {mlp_ckpt['hidden_dims']}")

plt.figure(figsize=(7, 4), dpi=100)
plt.plot(range(1, len(mlp_ckpt['loss_history']) + 1), mlp_ckpt['loss_history'], 'o-', color='#3498db')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Tabular MLP Training Loss Convergence')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 3. Sequence Modeling with Text Bi-LSTM
For natural language text in Portuguese review comments, sequence models process lexical semantics.

In [ ]:
bilstm_ckpt = torch.load(REPO_ROOT / 'artifacts' / 'models' / 'text_bilstm_review_sentiment.pt', weights_only=False)
print('Text Bi-LSTM Architecture Summary:')
print(f"- Vocab size: {len(bilstm_ckpt['word2idx'])}")
print(f"- Max sequence length: {bilstm_ckpt['max_seq_len']}")
print(f"- Embedding dimension: {bilstm_ckpt['embedding_dim']}")
print(f"- Hidden dimension: {bilstm_ckpt['hidden_dim']}")

plt.figure(figsize=(7, 4), dpi=100)
plt.plot(range(1, len(bilstm_ckpt['loss_history']) + 1), bilstm_ckpt['loss_history'], 's-', color='#e74c3c')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.title('Text Bi-LSTM Training Loss Convergence')
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()